# Body Controller Test Notebook

Live tuning and exploration of poses, antenna presets, beats, gestures, and the `orient` primitive.

Run cells top-to-bottom. The controller is instantiated **without** a state_bus — all motion is driven by direct method calls here, not the pipeline. No threads from BodyController are started unless you call `start()` in the state-animator section.

## 1. Setup

In [1]:
import sys, pathlib
# Make the repo root importable regardless of where Jupyter was launched
_repo_root = pathlib.Path.cwd().resolve()
if _repo_root.name == 'robot':
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import time, random, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s | %(message)s', datefmt='%H:%M:%S')

from reachy_mini import ReachyMini
from robot.components.body import (
    BodyController, Beat, Pose, POSES, ANTENNAS, GESTURES,
    MIN_JERK, CARTOON, EASE_IN_OUT, LINEAR,
)

mini = ReachyMini()
mini.__enter__()
bc = BodyController(mini=mini, state_bus=None)  # no state_bus → standalone test mode
bc.return_to_center()
print('connected, at center')

16:00:25 reachy_mini.reachy_mini | Connection mode selected: localhost_only
16:00:25 reachy_mini.reachy_mini | Auto-detected local IPC endpoint. Using LOCAL backend.
16:00:25 reachy_mini.media.media_manager | Using LOCAL backend (GStreamer IPC camera + GStreamer audio).

(<unknown>:17480): GStreamer-WARNING **: 16:00:25.584: External plugin loader failed. This most likely means that the plugin loader helper binary was not found or could not be run. You might need to set the GST_PLUGIN_SCANNER environment variable if your setup is unusual. This should normally not be required though.

(<unknown>:17480): GStreamer-WARNING **: 16:00:25.587: Failed to load plugin '/Users/aryanluthra/PersonalProjects/not-jarvis/.conda/lib/python3.11/site-packages/gstreamer_plugins/lib/gstreamer-1.0/libgstcurl.dylib': dlopen(/Users/aryanluthra/PersonalProjects/not-jarvis/.conda/lib/python3.11/site-packages/gstreamer_plugins/lib/gstreamer-1.0/libgstcurl.dylib, 0x0006): Symbol not found: _SSL_get0_group_name
 

connected, at center


## 2. What's available

In [2]:
print('POSES:    ', bc.list_poses())
print('ANTENNAS: ', bc.list_antennas())
print('GESTURES: ', bc.list_gestures())

POSES:     ['center', 'tilt_L', 'tilt_R', 'look_away_L', 'look_away_R', 'ponder_up_R', 'ponder_up_L', 'lean_in', 'pull_back', 'stretch_up', 'duck', 'recoil', 'peek_L', 'peek_R', 'nod_down', 'nod_up', 'look_up', 'gaze_L', 'gaze_R']
ANTENNAS:  ['relaxed', 'alert', 'perked', 'droop', 'listen_L', 'listen_R', 'curious_L', 'curious_R', 'flick_L', 'flick_R', 'flat']
GESTURES:  ['head_tilt_L', 'head_tilt_R', 'double_tilt', 'ear_flick_L', 'ear_flick_R', 'both_perk', 'slow_blink', 'settling_sigh', 'sharp_nod', 'lean_in', 'pull_back', 'alert_freeze', 'startled_recoil', 'gaze_shift_L', 'gaze_shift_R', 'look_away_L', 'look_away_R', 'curious_peek_L', 'curious_peek_R', 'weight_shift_L', 'weight_shift_R', 'preen_beat', 'yawn', 'stretch', 'scratch', 'sneeze', 'shake_off', 'self_groom_check', 'look_up', 'measuring_bob']


16:00:30 reachy_mini.media.camera_gstreamer | Pipeline latency (live=True, min_latency=20000000, max_latency=1000000000)


## 3. Test individual poses

Each pose is played as a single Beat. Tweak the pose in `poses.py` and re-import to see the effect.

In [4]:
# Single pose by name
bc.play_beat(Beat(pose='tilt_L', antennas='curious_L', duration=0.3, method=CARTOON, hold=1.0))
bc.return_to_center()

In [5]:
# Cycle through every pose (0.8s hold each)
for name in bc.list_poses():
    print(name, '→', POSES[name])
    bc.play_beat(Beat(pose=name, duration=0.3, method=MIN_JERK, hold=0.8))
bc.return_to_center()

center → Pose(yaw=2, pitch=3, roll=0, x=0, y=0, z=0)
tilt_L → Pose(yaw=6, pitch=3, roll=12, x=0, y=0.012, z=0)
tilt_R → Pose(yaw=-6, pitch=3, roll=-12, x=0, y=-0.012, z=0)
look_away_L → Pose(yaw=-14, pitch=-8, roll=-3, x=-0.015, y=0, z=0)
look_away_R → Pose(yaw=14, pitch=-8, roll=3, x=-0.015, y=0, z=0)
ponder_up_R → Pose(yaw=8, pitch=-6, roll=3, x=-0.01, y=0, z=0)
ponder_up_L → Pose(yaw=-8, pitch=-6, roll=-3, x=-0.01, y=0, z=0)
lean_in → Pose(yaw=0, pitch=-3, roll=0, x=0.02, y=0, z=0)
pull_back → Pose(yaw=0, pitch=5, roll=0, x=-0.025, y=0, z=0)
stretch_up → Pose(yaw=0, pitch=-3, roll=0, x=0, y=0, z=0.015)
duck → Pose(yaw=0, pitch=2, roll=0, x=0, y=0, z=-0.012)
recoil → Pose(yaw=0, pitch=6, roll=0, x=-0.03, y=0, z=0.006)
peek_L → Pose(yaw=4, pitch=0, roll=0, x=0, y=0.018, z=0)
peek_R → Pose(yaw=-4, pitch=0, roll=0, x=0, y=-0.018, z=0)
nod_down → Pose(yaw=0, pitch=7, roll=0, x=0.01, y=0, z=0)
nod_up → Pose(yaw=0, pitch=-3, roll=0, x=-0.005, y=0, z=0)
look_up → Pose(yaw=0, pitch=-15, roll

In [8]:
# Construct an ad-hoc pose inline (useful for eyeballing new amplitudes)
custom = Pose(yaw=20, pitch=-5, roll=8, x=0.02, y=0.01, z=0.015)
bc.play_beat(Beat(pose=custom, antennas='perked', duration=0.4, method=CARTOON, hold=1.0))
bc.return_to_center()

## 4. Test antenna presets

In [9]:
# Antenna-only moves (pose=None keeps head still)
for name in bc.list_antennas():
    print(name, '→', ANTENNAS[name])
    bc.play_beat(Beat(antennas=name, duration=0.2, method=CARTOON, hold=0.6))
bc.play_beat(Beat(antennas='relaxed', duration=0.3, method=MIN_JERK))

relaxed → [-0.1, -0.1]
alert → [0.3, 0.3]
perked → [0.5, 0.5]
droop → [-0.5, -0.5]
listen_L → [0.5, 0.1]
listen_R → [0.1, 0.5]
curious_L → [0.4, -0.1]
curious_R → [-0.1, 0.4]
flick_L → [0.6, -0.1]
flick_R → [-0.1, 0.6]
flat → [0.0, 0.0]


## 5. Test beats (all field combinations)

A Beat can touch any combination of three channels. Fields that are `None` keep the current value.

In [11]:
# Head only
bc.play_beat(Beat(pose='tilt_L', duration=0.3, method=CARTOON, hold=0.5))
# Antennas only (head stays tilted)
bc.play_beat(Beat(antennas='flick_R', duration=0.15, method=CARTOON, hold=0.3))
# Body only (head and antennas unchanged)
bc.play_beat(Beat(body_yaw=0.3, duration=0.4, method=MIN_JERK, hold=0.6))
# All three at once
bc.play_beat(Beat(pose='center', antennas='relaxed', body_yaw=0.0, duration=0.5, method=MIN_JERK))

In [12]:
# Compare interpolation methods on the same move
for method in [LINEAR, MIN_JERK, EASE_IN_OUT, CARTOON]:
    print('method:', method)
    bc.play_beat(Beat(pose='nod_down', duration=0.3, method=method, hold=0.2))
    bc.play_beat(Beat(pose='center',  duration=0.3, method=method, hold=0.4))
    bc.play_beat(Beat(pose='tilt_L', duration=0.3, method=CARTOON, hold=0.5))
    bc.play_beat(Beat(pose='tilt_R', duration=0.3, method=CARTOON, hold=0.5))
    bc.play_beat(Beat(pose='center',  duration=0.3, method=method, hold=0.4))


method: InterpolationTechnique.LINEAR
method: InterpolationTechnique.MIN_JERK
method: InterpolationTechnique.EASE_IN_OUT
method: InterpolationTechnique.CARTOON


## 6. Test gestures

In [13]:
# Play a single gesture by name
bc.play_gesture('yawn')
bc.return_to_center()

In [14]:
# Demo every gesture with a 1.5s pause between. Ctrl+C to stop early.
for name in bc.list_gestures():
    print('▶', name)
    bc.play_gesture(name)
    time.sleep(1.5)
bc.return_to_center()

▶ head_tilt_L
▶ head_tilt_R
▶ double_tilt
▶ ear_flick_L
▶ ear_flick_R
▶ both_perk
▶ slow_blink
▶ settling_sigh
▶ sharp_nod
▶ lean_in
▶ pull_back
▶ alert_freeze
▶ startled_recoil
▶ gaze_shift_L
▶ gaze_shift_R
▶ look_away_L
▶ look_away_R
▶ curious_peek_L
▶ curious_peek_R
▶ weight_shift_L
▶ weight_shift_R
▶ preen_beat
▶ yawn
▶ stretch
▶ scratch
▶ sneeze
▶ shake_off
▶ self_groom_check
▶ look_up
▶ measuring_bob


In [22]:
# Inspect what lives in each state's pool
from robot.components.body.gestures import IDLE_GESTURES, LISTENING_GESTURES, THINKING_GESTURES, SPEAKING_GESTURES, TIER
for label, pool in [('IDLE', IDLE_GESTURES), ('LISTENING', LISTENING_GESTURES), ('THINKING', THINKING_GESTURES), ('SPEAKING', SPEAKING_GESTURES)]:
    print(f'{label}:')
    for g in pool:
        print(f'  {TIER.get(g, "common"):10s}  {g}')

IDLE:
  common      gaze_shift_L
  common      gaze_shift_R
  common      weight_shift_L
  common      weight_shift_R
  common      preen_beat
  uncommon    head_tilt_L
  uncommon    head_tilt_R
  uncommon    double_tilt
  uncommon    curious_peek_L
  uncommon    curious_peek_R
  uncommon    slow_blink
  uncommon    both_perk
  rare        settling_sigh
  rare        look_up
  rare        measuring_bob
  rare        alert_freeze
  rare        self_groom_check
  very_rare   yawn
  very_rare   stretch
  very_rare   scratch
  very_rare   sneeze
  very_rare   shake_off
  very_rare   startled_recoil
LISTENING:
  common      ear_flick_L
  common      ear_flick_R
  uncommon    head_tilt_L
  uncommon    head_tilt_R
  uncommon    lean_in
THINKING:
  very_rare   look_away_L
  very_rare   look_away_R
  uncommon    slow_blink
SPEAKING:
  common      sharp_nod
  common      weight_shift_L
  common      weight_shift_R
  common      ear_flick_L
  common      ear_flick_R
  uncommon    head_tilt_L
  un

## 7. Test `orient` (body/head split + phasing)

- **< 10°** → head only, no body
- **10–25°** → 80% head, 20% body, 50 ms head lead
- **25–40°** → 60% head, 40% body, 90 ms head lead
- **> 40°** → 50% head, 50% body, 120 ms head lead

In [16]:
# Sweep angles in both directions — watch where body starts kicking in
for angle in [5, -5, 15, -15, 30, -30, 45, -45, 60, -60, 90, -90]:
    print(f'orient({angle:+d}°)')
    bc.orient(angle)
    time.sleep(1.2)
    bc.return_to_center(duration=0.5)
    time.sleep(0.4)

orient(+5°)
orient(-5°)
orient(+15°)
orient(-15°)
orient(+30°)
orient(-30°)
orient(+45°)
orient(-45°)
orient(+60°)
orient(-60°)
orient(+90°)
orient(-90°)


## 8. Test state animators

Spin up a real `Bus` and publish `StateChange` events to drive the animators. This is the closest to the pipeline behavior — stop the cell or push a new state to transition.

In [17]:
from robot.core import Bus, StateChange

# Tear down any previous controller
try:
    bc.stop()
except Exception:
    pass

state_bus = Bus()
bc = BodyController(mini=mini, state_bus=state_bus)
bc.start()
print('body controller started, current state: idle')

16:07:03 reachy_mini.reachy_mini | Move cancellation requested


body controller started, current state: idle


In [18]:
# Flip through states — watch the transition beats and per-state vibe
for state in ['listening', 'thinking', 'speaking', 'idle']:
    print(f'→ {state}')
    state_bus.put(StateChange(state=state))
    time.sleep(8)  # dwell in each state

16:07:05 reachy_mini.reachy_mini | Move cancellation requested


→ listening


16:07:13 reachy_mini.reachy_mini | Move cancellation requested


→ thinking


16:07:21 reachy_mini.reachy_mini | Move cancellation requested


→ speaking


16:07:29 reachy_mini.reachy_mini | Move cancellation requested


→ idle


In [20]:
bc.return_to_center()
bc.stop()
print('body controller stopped')

16:09:59 reachy_mini.reachy_mini | Move cancellation requested


body controller stopped


18:18:31 reachy_mini.media.camera_gstreamer | GStreamer pipeline error (domain=gst-stream-error-quark, code=1): Internal data stream error.
